# Fast Tokenizers' Special Powers

**Source:** [HuggingFace LLM Course – Chapter 6, Section 3](https://huggingface.co/learn/llm-course/chapter6/3)

In this notebook, we explore the special capabilities of 'fast' tokenizers, which are written in Rust.
Beyond being extremely fast, they provide crucial features like **offset mapping**, which allows us to map generated tokens directly back to the original text characters. This is essential for tasks like Token Classification (NER) and Question Answering.

---

## Step 1: Install Required Libraries

We install the necessary libraries for tokenization and model evaluation.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

## Step 2: Load a Fast Tokenizer

Here we load a BERT tokenizer and encode a sample sentence. Notice that the output `encoding` is a `BatchEncoding` object. By default, `AutoTokenizer` tries to load a 'fast' tokenizer backed by the Rust Hugging Face Tokenizers library.

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
example = "My name is Sujat and I work at Blah Blah in Toronto."
encoding = tokenizer(example)
print(type(encoding))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


## Step 3: Verify Tokenizer Type

We can check if our tokenizer is actually a fast tokenizer by inspecting the `.is_fast` attribute.

In [4]:
tokenizer.is_fast

True

## Step 4: Verify Encoding Type

Similarly, the `BatchEncoding` object also has an `.is_fast` attribute indicating it was generated by a fast tokenizer.

In [5]:
encoding.is_fast

True

## Step 5: View the Tokens

Fast tokenizers allow us to easily retrieve the list of tokens generated from the input text without needing to call a separate method.

In [6]:
encoding.tokens()

['[CLS]',
 'My',
 'name',
 'is',
 'Su',
 '##ja',
 '##t',
 'and',
 'I',
 'work',
 'at',
 'B',
 '##lah',
 'B',
 '##lah',
 'in',
 'Toronto',
 '.',
 '[SEP]']

## Step 6: Map Tokens to Word IDs

One of the special powers of fast tokenizers is offset mapping. Here we use `word_ids()` to map each token back to the index of the word it came from in the original sentence. `None` represents special tokens like `[CLS]` and `[SEP]`.

In [7]:
encoding.word_ids()

[None, 0, 1, 2, 3, 3, 3, 4, 5, 6, 7, 8, 8, 9, 9, 10, 11, 12, None]

## Step 7: Map Words to Characters

We can also go further and map a specific word index (like word 3, which is 'Sujat') back to the exact character span `(start, end)` in the original text string.

In [8]:
start, end = encoding.word_to_chars(3)
example[start:end]

'Sujat'

## Step 8: Standard Token Classification Pipeline

Let's see why offset mapping is useful! If we run a standard NER (Named Entity Recognition) pipeline, the model predicts entities for individual tokens (e.g., 'Su', '##ja', '##t').

In [10]:
from transformers import pipeline

token_classifier = pipeline("token-classification")
token_classifier("My name is Sujat and I work at Blah Blah in Toronto.")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity': 'I-PER',
  'score': np.float32(0.9988991),
  'index': 4,
  'word': 'Su',
  'start': 11,
  'end': 13},
 {'entity': 'I-PER',
  'score': np.float32(0.994429),
  'index': 5,
  'word': '##ja',
  'start': 13,
  'end': 15},
 {'entity': 'I-PER',
  'score': np.float32(0.99693215),
  'index': 6,
  'word': '##t',
  'start': 15,
  'end': 16},
 {'entity': 'I-ORG',
  'score': np.float32(0.99697614),
  'index': 11,
  'word': 'B',
  'start': 31,
  'end': 32},
 {'entity': 'I-ORG',
  'score': np.float32(0.9890994),
  'index': 12,
  'word': '##lah',
  'start': 32,
  'end': 35},
 {'entity': 'I-ORG',
  'score': np.float32(0.99158067),
  'index': 13,
  'word': 'B',
  'start': 36,
  'end': 37},
 {'entity': 'I-ORG',
  'score': np.float32(0.9894406),
  'index': 14,
  'word': '##lah',
  'start': 37,
  'end': 40},
 {'entity': 'I-LOC',
  'score': np.float32(0.998323),
  'index': 16,
  'word': 'Toronto',
  'start': 44,
  'end': 51}]

## Step 9: Pipeline with Aggregation Strategy

By passing `aggregation_strategy="simple"`, the pipeline uses the fast tokenizer's offset mapping to group sub-tokens back into full words and output clean entities (e.g., 'Sujat' as a single PER entity).

In [11]:
from transformers import pipeline

token_classifier = pipeline("token-classification", aggregation_strategy="simple")
token_classifier("My name is Sujat and I work at Blah Blah in Toronto.")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9967534),
  'word': 'Sujat',
  'start': 11,
  'end': 16},
 {'entity_group': 'ORG',
  'score': np.float32(0.9917742),
  'word': 'Blah Blah',
  'start': 31,
  'end': 40},
 {'entity_group': 'LOC',
  'score': np.float32(0.998323),
  'word': 'Toronto',
  'start': 44,
  'end': 51}]

## Step 10: Manual Token Classification (Without Pipeline)

To understand how the pipeline works under the hood, let's load the model and tokenizer manually and process our example sentence.

In [12]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)

example = "My name is Sujat and I work at Blah Blah in Toronto."
inputs = tokenizer(example, return_tensors="pt")
outputs = model(**inputs)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 11: Inspect Model Outputs

The input is a sequence of 19 tokens. The model outputs logits for each of these 19 tokens across 9 possible NER classes.

In [13]:
print(inputs["input_ids"].shape)
print(outputs.logits.shape)

torch.Size([1, 19])
torch.Size([1, 19, 9])


## Step 12: Get Predictions and Probabilities

We apply a softmax to the logits to get probabilities, and `argmax` to get the index of the predicted class for each token.

In [14]:
import torch

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()
predictions = outputs.logits.argmax(dim=-1)[0].tolist()
print(predictions)

[0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 6, 6, 6, 6, 0, 8, 0, 0]


## Step 13: View Label Mapping

The model config contains the mapping from predicted indices (0-8) to actual human-readable NER tags (like `B-PER`, `I-ORG`, etc.).

In [15]:
model.config.id2label

{0: 'O',
 1: 'B-MISC',
 2: 'I-MISC',
 3: 'B-PER',
 4: 'I-PER',
 5: 'B-ORG',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'I-LOC'}

## Step 14: Reconstruct Entities (Naive Approach)

If we just map the predictions to labels for each token, we get the fragmented sub-tokens (e.g., 'Su', '##ja', '##t') just like the basic pipeline. This loses the original formatting.

In [16]:
results = []
tokens = inputs.tokens()

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        results.append(
            {"entity": label, "score": probabilities[idx][pred], "word": tokens[idx]}
        )

print(results)

[{'entity': 'I-PER', 'score': 0.9988991022109985, 'word': 'Su'}, {'entity': 'I-PER', 'score': 0.9944289922714233, 'word': '##ja'}, {'entity': 'I-PER', 'score': 0.9969319105148315, 'word': '##t'}, {'entity': 'I-ORG', 'score': 0.9969761371612549, 'word': 'B'}, {'entity': 'I-ORG', 'score': 0.989099383354187, 'word': '##lah'}, {'entity': 'I-ORG', 'score': 0.9915806651115417, 'word': 'B'}, {'entity': 'I-ORG', 'score': 0.9894406199455261, 'word': '##lah'}, {'entity': 'I-LOC', 'score': 0.9983230233192444, 'word': 'Toronto'}]


## Step 15: Generate Offset Mapping

To fix this, we need the fast tokenizer's offset mapping! Passing `return_offsets_mapping=True` gives us a list of `(start, end)` character tuples for every token.

In [17]:
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
inputs_with_offsets["offset_mapping"]

[(0, 0),
 (0, 2),
 (3, 7),
 (8, 10),
 (11, 13),
 (13, 15),
 (15, 16),
 (17, 20),
 (21, 22),
 (23, 27),
 (28, 30),
 (31, 32),
 (32, 35),
 (36, 37),
 (37, 40),
 (41, 43),
 (44, 51),
 (51, 52),
 (0, 0)]

## Step 16: Verify the Offsets

We can verify that slicing the original string with the offset tuple `(12, 14)` correctly returns the exact characters for that sub-token.

In [18]:
example[12:14]

'uj'

## Step 17: Reconstruct Entities with Offsets

Now, instead of outputting the sub-token string (like '##lah'), we use the `(start, end)` offsets to grab the exact substring from the original text!

In [19]:
results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        start, end = offsets[idx]
        results.append(
            {
                "entity": label,
                "score": probabilities[idx][pred],
                "word": tokens[idx],
                "start": start,
                "end": end,
            }
        )

print(results)

[{'entity': 'I-PER', 'score': 0.9988991022109985, 'word': 'Su', 'start': 11, 'end': 13}, {'entity': 'I-PER', 'score': 0.9944289922714233, 'word': '##ja', 'start': 13, 'end': 15}, {'entity': 'I-PER', 'score': 0.9969319105148315, 'word': '##t', 'start': 15, 'end': 16}, {'entity': 'I-ORG', 'score': 0.9969761371612549, 'word': 'B', 'start': 31, 'end': 32}, {'entity': 'I-ORG', 'score': 0.989099383354187, 'word': '##lah', 'start': 32, 'end': 35}, {'entity': 'I-ORG', 'score': 0.9915806651115417, 'word': 'B', 'start': 36, 'end': 37}, {'entity': 'I-ORG', 'score': 0.9894406199455261, 'word': '##lah', 'start': 37, 'end': 40}, {'entity': 'I-LOC', 'score': 0.9983230233192444, 'word': 'Toronto', 'start': 44, 'end': 51}]


## Step 18: The Problem with Simple Offsets

Wait, there's still a small issue. If we look at the character span from the first 'B' to the last '##lah' in 'Blah Blah', we get extra characters or spaces if we just naively concatenate them.

In [20]:
example[33:45]

'ah Blah in T'

## Step 19: Full Entity Aggregation

To perfectly recreate the `aggregation_strategy="simple"`, we write a loop that groups consecutive tokens belonging to the same entity (e.g., `B-ORG` followed by `I-ORG`), averages their scores, and uses the start offset of the first token and the end offset of the last token to slice the clean word from the original text!

In [21]:
import numpy as np

results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]

idx = 0
while idx < len(predictions):
    pred = predictions[idx]
    label = model.config.id2label[pred]
    if label != "O":
        # Remove the B- or I-
        label = label[2:]
        start, _ = offsets[idx]

        # Grab all the tokens labeled with I-label
        all_scores = []
        while (
            idx < len(predictions)
            and model.config.id2label[predictions[idx]] == f"I-{label}"
        ):
            all_scores.append(probabilities[idx][pred])
            _, end = offsets[idx]
            idx += 1

        # The score is the mean of all the scores of the tokens in that grouped entity
        score = np.mean(all_scores).item()
        word = example[start:end]
        results.append(
            {
                "entity_group": label,
                "score": score,
                "word": word,
                "start": start,
                "end": end,
            }
        )
    idx += 1

print(results)

[{'entity_group': 'PER', 'score': 0.9967533349990845, 'word': 'Sujat', 'start': 11, 'end': 16}, {'entity_group': 'ORG', 'score': 0.9917742013931274, 'word': 'Blah Blah', 'start': 31, 'end': 40}, {'entity_group': 'LOC', 'score': 0.9983230233192444, 'word': 'Toronto', 'start': 44, 'end': 51}]
